# Reproducibility patch — 2026-10-12

This is a **versioned execution-code correction**, not a fresh model inference run. The immutable historical notebook remains in the adjacent dated file. The earlier phrase *pre-registered deterministic variants* and original unexecuted-session note were pre-run planning comments; the sensitivity experiments were subsequently completed as **post-hoc** audits. This copy pins the Mistral-7B-Instruct-v0.3 checkpoint at revision `c170c708c41dac9275d15a8fff4eca08d52bab71`, verified against the frozen `mistral_fp16_offload_pinned.json` server execution manifest. The original initial reference FP16 T1 invocation is distinct from the E1/rot0 matched baseline; inspect the 2026-10 baseline audit before comparing results.

No new predictions or experiments are claimed by releasing this patch.


# P2 — Judge Answer-Code and Option-Order Sensitivity (Google Colab / GPU)

**Purpose (not an architecture or model-family comparison):** assess how stable the *semantic failure judgments* are under controlled changes in (E1) answer-code assignment, and (E2) display order of rubric options, separately for Qwen3-8B and Mistral-7B-Instruct-v0.3.

**Same as supplied P2 notebook:** frozen FINAL GOLD + OOF baselines; T1–T4; evidence condition C; FP16 unquantized model weights; original target definitions; closed-choice next-token log-probability scoring; Qwen system/user messages with thinking disabled; Mistral single user message. This is a NEW sensitivity study, not an architecture comparison, and is **not yet executed**.

### Required uploads
* `P2_FINAL_SENIOR_ADJUDICATED_GOLD_v1.0.xlsx` (sheet `FINAL_GOLD`)
* `P2_CPU_Baselines_v1.0.xlsx` (sheets `OOF_T1`, `OOF_T2`, `OOF_T3`, `OOF_T4`)

Default paths: Google Drive `MyDrive/P2/P2/inputs/` (editable below). Each target/variant writes an append-only CSV after **each individual case**, permitting resume. Downloadable ZIP contains `results/`, `reports/`, `manifest/`.

**Run order:** 1 Mount/install → 2 Verify data → 3 Inspect plans → 4 Run one judge → 5 Repeat for second judge → 6 Export. In a new Colab session, rerun cells 1–3 and then resume cell 4. Prefer high-VRAM Colab GPU (A100 or 40+ GiB); on T4, 7B/8B FP16 may require CPU/disk offload and is **not guaranteed** to fit. Offload can be slow; no quantization fallback is performed.


In [ ]:
%pip -q install "transformers==4.57.6" "huggingface_hub==0.36.2" "accelerate>=1.0" "safetensors" "pandas" "openpyxl" "scikit-learn>=1.4,<2" "psutil" "nbformat"
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, json, hashlib, gc, time, zipfile, shutil, platform, inspect, random
from itertools import permutations
import numpy as np, pandas as pd
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, cohen_kappa_score, balanced_accuracy_score
import torch, transformers
from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModelForCausalLM

ROOT = Path('/content/drive/MyDrive/P2/P2')  # change to your actual project folder
INPUT = ROOT / 'inputs'                         # exact frozen inputs; do NOT use prediction outputs as GOLD
RUN_ID = 'JUDGE_SENSITIVITY_20261009_A'
RUN = ROOT / 'results' / RUN_ID
RAW, REPORTS, MANIFEST = (RUN/x for x in ('results','reports','manifest'))
for x in (RAW,REPORTS,MANIFEST): x.mkdir(parents=True,exist_ok=True)
LOCAL_OFFLOAD = Path('/content/p2_sensitivity_offload'); LOCAL_OFFLOAD.mkdir(exist_ok=True)
# Adjust this for your GPU. Examples: 9 for T4 16GB (offload required), 32 for 40GB.
GPU_MAX_MEMORY_GIB = 9 if torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory/2**30 < 24 else 30
CPU_MAX_MEMORY_GIB = 3
MAX_LENGTH = 4096
TARGETS = ['T1','T2','T3','T4']
JUDGES = {
 'mistral_fp16_offload': {'model_id':'mistralai/Mistral-7B-Instruct-v0.3','revision':'c170c708c41dac9275d15a8fff4eca08d52bab71'},
 'qwen_fp16_offload': {'model_id':'Qwen/Qwen3-8B','revision':'e8bbd8252970581ea5b08b6a5b3e668adaf3161a'}
}
# Start with Mistral. Change to qwen_fp16_offload in a fresh session for the second judge.
JUDGE_TO_RUN = 'mistral_fp16_offload'
MAX_NEW_CASES = None  # set e.g. 80 to limit work per Colab session. None runs all missing cases.
SEED = 20261009
assert torch.cuda.is_available(), 'GPU runtime required: Runtime > Change runtime type > GPU'
print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
print('Input:',INPUT,'Output:',RUN)


In [ ]:
GOLD_NAME='P2_FINAL_SENIOR_ADJUDICATED_GOLD_v1.0.xlsx'
BASE_NAME='P2_CPU_Baselines_v1.0.xlsx'
EXPECTED_SHA={GOLD_NAME:'ccc910b7bafbfa9607e93ef2eba8605f56f7ed87460ef74892d4877c7db4de65',
 BASE_NAME:'4502e71bd6d1b0b10650caaa433d7188bf958017d2fc646d328a31163'}
EXPECTED_N={'T1':720,'T2':474,'T3':460,'T4':474}
def sha256(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for chunk in iter(lambda:f.read(2**20),b''):h.update(chunk)
 return h.hexdigest()
for fn,expected in EXPECTED_SHA.items():
 p=INPUT/fn
 assert p.is_file(),f'Missing exact input: {p}'
 actual=sha256(p);print(fn,actual)
 assert actual==expected,'Input SHA-256 mismatch: wrong version. Do not bypass.'
gold=pd.read_excel(INPUT/GOLD_NAME,sheet_name='FINAL_GOLD').reset_index(drop=True)
assert len(gold)==720 and gold.Case_ID.is_unique and gold.Prompt_ID.nunique()==120
GOLD=gold.assign(Case_ID=gold.Case_ID.astype(str)).set_index('Case_ID',drop=False)
oof={}
for t in TARGETS:
 d=pd.read_excel(INPUT/BASE_NAME,sheet_name='OOF_'+t)
 d['Case_ID']=d.Case_ID.astype(str)
 assert len(d)==EXPECTED_N[t] and d.Case_ID.is_unique and set(d.Case_ID).issubset(GOLD.index)
 assert set(d.y_true.astype(int))==({1,2,3} if t=='T4' else {0,1})
 assert all(d[['Case_ID','Prompt_ID']].merge(gold[['Case_ID','Prompt_ID']],on='Case_ID',suffixes=('_oof','_gold')).eval('Prompt_ID_oof == Prompt_ID_gold'))
 oof[t]=d.sort_values('Case_ID',kind='stable').reset_index(drop=True)
print({t:len(d) for t,d in oof.items()})


In [ ]:
RUBRICS = {'T1': {'codes': ['A', 'B'], 'map': {'A': 0, 'B': 1}, 'rubric': 'Decide only whether the response contains an identifiable semantic-coherence or authorised-grounding failure.\nA = no_failure: no identifiable failure under the supplied evidence.\nB = failure: at least one identifiable semantic-coherence or authorised-grounding failure.\nDo not classify the subtype. Choose only A or B.'}, 'T2': {'codes': ['A', 'B'], 'map': {'A': 0, 'B': 1}, 'rubric': 'The case is already known to contain a failure. Decide whether authorised grounding is involved.\nA = semantic_only: the failure is meaning-level but does not require unsupported/fabricated content or conflict with an authorised visible anchor.\nB = grounding_involved: the failure includes overlap with unsupported/fabricated content or authorised-anchor conflict, including hallucination-only cases.\nChoose only A or B.'}, 'T3': {'codes': ['A', 'B'], 'map': {'A': 0, 'B': 1}, 'rubric': 'The case belongs to one of two mechanism families.\nA = Core_SCF: topic/QUD shift, causal-chain failure, response-goal misalignment, discourse-structure failure, local-to-global breakdown, referential instability, or term-sense instability.\nB = OVR: self-contradiction, visible-context inconsistency, entity/role/attribute misattribution, temporal/world-state inconsistency, reasoning-answer mismatch, or mixed-context synthesis failure.\nChoose only A or B.'}, 'T4': {'codes': ['A', 'B', 'C'], 'map': {'A': 1, 'B': 2, 'C': 3}, 'rubric': 'The case is already known to contain a failure. Judge functional severity, not confidence.\nA = severity 1 (Minor): localized disruption with limited effect on the main answer.\nB = severity 2 (Moderate): materially weakens an important task-relevant relation or component.\nC = severity 3 (Severe): undermines the main answer, reverses meaning, contradicts the prompt/source, or makes the answer unreliable for the task.\nChoose only A, B, or C.'}}


In [ ]:
# Stable, post-hoc deterministic variants (not preregistered); E1 and E2 DO NOT CHANGE TOGETHER.
# E1: order stays canonical, rotate the category-to-answer-code mapping.
# E2: category-to-answer-code mapping stays canonical, rotate visible option line order.
# T1/T2/T3: 2 mappings + 2 orders (the identity variants are intentionally shared).
# T4: 3 rotations for each experiment (rotations only, not all 6 permutations).

def variants(target):
 codes=RUBRICS[target]['codes']; n=len(codes)
 canonical_labels=[RUBRICS[target]['map'][c] for c in codes]
 out=[]
 for shift in range(n):
  out.append({'experiment':'E1_CODE','variant':f'code_rot{shift}',
    'label_to_code':{str(l):codes[(i+shift)%n] for i,l in enumerate(canonical_labels)},
    'display_order':canonical_labels[:]})
 for shift in range(1,n):
  out.append({'experiment':'E2_ORDER','variant':f'order_rot{shift}',
    'label_to_code':{str(l):codes[i] for i,l in enumerate(canonical_labels)},
    'display_order':canonical_labels[shift:]+canonical_labels[:shift]})
 return out
for t in TARGETS:print(t,variants(t))

SYSTEM = 'You are a conservative semantic-coherence evaluator.\nApply only the supplied target rubric to the visible text.\nDo not infer hidden facts. Do not output an explanation.\nYour decision space is closed: select exactly one of the permitted answer codes.'

def serialise_C(g):
 prompt=str(g['Prompt_Text'] if pd.notna(g['Prompt_Text']) else '').strip()
 source=str(g['Source_Context'] if pd.notna(g['Source_Context']) else '').strip()
 output=str(g['Model_Output'] if pd.notna(g['Model_Output']) else '').strip()
 if str(g['Source_Available']).lower()=='yes' and source:
  return f'Task:\n{prompt}\n\nAuthorised source/context:\n{source}\n\nResponse:\n{output}'
 return f'Task:\n{prompt}\n\nResponse:\n{output}'

# Semantic descriptions are taken directly from the supplied original rubrics.
LABEL_TEXT={
 'T1':{0:'no_failure: no identifiable failure under the supplied evidence.',1:'failure: at least one identifiable semantic-coherence or authorised-grounding failure.'},
 'T2':{0:'semantic_only: the failure is meaning-level but does not require unsupported/fabricated content or conflict with an authorised visible anchor.',1:'grounding_involved: the failure includes overlap with unsupported/fabricated content or authorised-anchor conflict, including hallucination-only cases.'},
 'T3':{0:'Core_SCF: topic/QUD shift, causal-chain failure, response-goal misalignment, discourse-structure failure, local-to-global breakdown, referential instability, or term-sense instability.',1:'OVR: self-contradiction, visible-context inconsistency, entity/role/attribute misattribution, temporal/world-state inconsistency, reasoning-answer mismatch, or mixed-context synthesis failure.'},
 'T4':{1:'severity 1 (Minor): localized disruption with limited effect on the main answer.',2:'severity 2 (Moderate): materially weakens an important task-relevant relation or component.',3:'severity 3 (Severe): undermines the main answer, reverses meaning, contradicts the prompt/source, or makes the answer unreliable for the task.'}
}
PREAMBLE={
 'T1':'Decide only whether the response contains an identifiable semantic-coherence or authorised-grounding failure.',
 'T2':'The case is already known to contain a failure. Decide whether authorised grounding is involved.',
 'T3':'The case belongs to one of two mechanism families.',
 'T4':'The case is already known to contain a failure. Judge functional severity, not confidence.'
}
TAIL={
 'T1':'Do not classify the subtype.',
 'T2':'',
 'T3':'',
 'T4':''
}
def rubric_variant(target,var):
 lines=[PREAMBLE[target]]
 for label in var['display_order']:
  lines.append(f"{var['label_to_code'][str(label)]} = {LABEL_TEXT[target][label]}")
 if TAIL[target]:lines.append(TAIL[target])
 allowed=RUBRICS[target]['codes']
 lines.append('Choose only '+(' or '.join(allowed) if len(allowed)==2 else ', '.join(allowed[:-1])+', or '+allowed[-1])+'.')
 return '\n'.join(lines)

def judge_chat(g,target,var,tokenizer,judge_name):
 user=rubric_variant(target,var)+'\n\nVisible case:\n'+serialise_C(g)
 if judge_name.startswith('qwen_'):
  messages=[{'role':'system','content':SYSTEM},{'role':'user','content':user}]
 else:messages=[{'role':'user','content':SYSTEM+'\n\n'+user}]
 kw={'tokenize':False,'add_generation_prompt':True}
 if judge_name.startswith('qwen_'):kw['enable_thinking']=False
 return tokenizer.apply_chat_template(messages,**kw)


## 4. Execute judge — run separately for each model

Set `JUDGE_TO_RUN` above. Each variant is resumable from CSV; completed rows are never regenerated. Strict checks guard the input versions, model revision and exact experimental configuration. If Colab disconnects, remount Drive and rerun. **GPU session not run during notebook creation.**

In [ ]:
def atomic_write_json(obj,path):
 tmp=path.with_suffix('.tmp');tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False));os.replace(tmp,path)
def atomic_csv(df,path):
 tmp=path.with_suffix('.tmp');df.to_csv(tmp,index=False);os.replace(tmp,path)

def load_judge():
 assert JUDGE_TO_RUN in JUDGES
 cfg=JUDGES[JUDGE_TO_RUN].copy()
 rev=cfg['revision'] or HfApi().model_info(cfg['model_id']).sha
 assert len(rev)==40
 cfg['revision']=rev
 out=RAW/JUDGE_TO_RUN;out.mkdir(parents=True,exist_ok=True)
 pin=MANIFEST/f'{JUDGE_TO_RUN}_pinned.json'
 pin_contents={'model_id':cfg['model_id'],'revision':rev,'dtype':'float16','quantization':'none',
   'condition':'C','max_length':MAX_LENGTH,'gold_sha256':EXPECTED_SHA[GOLD_NAME],
   'base_sha256':EXPECTED_SHA[BASE_NAME],'seed':SEED,'transformers':transformers.__version__}
 if pin.exists():
  old=json.loads(pin.read_text())
  for key in ['model_id','revision','dtype','quantization','condition','max_length','gold_sha256','base_sha256','seed']:
   assert old[key]==pin_contents[key], f'Configuration changed: {key}'
 else:atomic_write_json(pin_contents,pin)
 tokenizer=AutoTokenizer.from_pretrained(cfg['model_id'],revision=rev)
 tokenizer.padding_side='left'
 if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token
 code_ids={}
 for c in 'ABC':
  ids=tokenizer(c,add_special_tokens=False)['input_ids']
  assert len(ids)==1, f'Answer code {c} is NOT single-token: {ids}'
  code_ids[c]=ids[0]
 free,total=torch.cuda.mem_get_info()
 print('Free VRAM GiB',free/2**30, 'Local offload storage GiB',shutil.disk_usage('/content').free/2**30)
 if free/2**30 < 7:raise RuntimeError('Insufficient free GPU memory; restart runtime.')
 if shutil.disk_usage('/content').free/2**30 < 20:raise RuntimeError('Insufficient local disk for model/offload.')
 model=AutoModelForCausalLM.from_pretrained(cfg['model_id'],revision=rev,torch_dtype=torch.float16,
      device_map='auto',max_memory={0:f'{GPU_MAX_MEMORY_GIB}GiB','cpu':f'{CPU_MAX_MEMORY_GIB}GiB'},
      low_cpu_mem_usage=True,offload_folder=str(LOCAL_OFFLOAD/JUDGE_TO_RUN),
      offload_state_dict=True,offload_buffers=True,attn_implementation='sdpa')
 assert not getattr(model,'is_loaded_in_4bit',False) and not getattr(model,'is_loaded_in_8bit',False)
 model.eval(); cfg['hf_device_map']={k:str(v) for k,v in model.hf_device_map.items()}
 atomic_write_json({**pin_contents,'device_map':cfg['hf_device_map'],'answer_code_token_ids':code_ids},MANIFEST/f'{JUDGE_TO_RUN}_loaded.json')
 return cfg,tokenizer,model,code_ids

@torch.inference_mode()
def predict_one(model,tokenizer,code_ids,prompt,allowed_codes):
 tokens=tokenizer(prompt,return_tensors='pt',add_special_tokens=False)
 token_count=int(tokens['input_ids'].shape[1])
 # Stop instead of silently truncating; ensure prompt, labels, and scoring position are unchanged.
 if token_count > MAX_LENGTH:
  return None, None, token_count, 'too_long'
 dev=torch.device('cuda:0')
 inputs={k:v.to(dev) for k,v in tokens.items()}
 kw={'use_cache':False}
 if 'logits_to_keep' in inspect.signature(model.forward).parameters:kw['logits_to_keep']=1
 elif 'num_logits_to_keep' in inspect.signature(model.forward).parameters:kw['num_logits_to_keep']=1
 else:raise RuntimeError('Model does not support last-token logits; avoid full-vocab logits memory spike.')
 logits=model(**inputs,**kw).logits[:,-1,:].float()
 score=torch.log_softmax(logits,dim=-1)[0]
 scores={c:float(score[code_ids[c]].cpu()) for c in allowed_codes}
 pred_code=max(scores,key=scores.get)
 return pred_code,scores,token_count,'ok'

def execute_judge():
 cfg,tokenizer,model,code_ids=load_judge()
 new_cases=0
 try:
  for target in TARGETS:
   d=oof[target]
   for var in variants(target):
    name=f"{target}_{var['experiment']}_{var['variant']}"
    path=RAW/JUDGE_TO_RUN/(name+'.csv')
    identity={'judge':JUDGE_TO_RUN,'model_revision':cfg['revision'],'target':target,'variant':var,'condition':'C',
      'gold_hash':EXPECTED_SHA[GOLD_NAME],'base_hash':EXPECTED_SHA[BASE_NAME]}
    signature=hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()
    old=pd.read_csv(path,dtype={'Case_ID':str}) if path.exists() else pd.DataFrame()
    if len(old):
     assert old.Case_ID.is_unique and set(old.Case_ID).issubset(set(d.Case_ID))
     assert set(old.signature.astype(str))=={signature},f'Config mismatch {path}'
    completed=set(old.Case_ID.astype(str)) if len(old) else set()
    print('WORK',JUDGE_TO_RUN,name,len(completed),'/',len(d),flush=True)
    for r in d.itertuples(index=False):
     cid=str(r.Case_ID)
     if cid in completed:continue
     if MAX_NEW_CASES is not None and new_cases>=MAX_NEW_CASES:
      print('PAUSED: session quota',new_cases);return
     prompt=judge_chat(GOLD.loc[cid],target,var,tokenizer,JUDGE_TO_RUN)
     pred_code,scores,n_tokens,status=predict_one(model,tokenizer,code_ids,prompt,RUBRICS[target]['codes'])
     inv={c:int(l) for l,c in var['label_to_code'].items()}
     pred=inv[pred_code] if status=='ok' else None
     record={'Case_ID':cid,'Prompt_ID':r.Prompt_ID,'Fold':int(r.Fold),'y_true':int(r.y_true),
        'pred':pred,'pred_code':pred_code,'status':status,'token_count':n_tokens,
        'scores':json.dumps(scores) if scores else '', 'signature':signature,
        'experiment':var['experiment'],'variant':var['variant'],'condition':'C',
        'model_revision':cfg['revision'],'model_id':cfg['model_id'],
        'input_sha256':hashlib.sha256(prompt.encode()).hexdigest()}
     old=pd.concat([old,pd.DataFrame([record])],ignore_index=True)
     atomic_csv(old,path)
     new_cases+=1
    print('SAVED',path,len(old),flush=True)
 finally:
  del model,tokenizer
  gc.collect();torch.cuda.empty_cache()
 print('FINISHED',JUDGE_TO_RUN,new_cases)

execute_judge()


## 5. Reports — per-class results, stability vs canonical mapping, manifest

Runs on CPU; no GPU model is loaded. The identity variant `E1_CODE/code_rot0` is the **newly rerun** canonical control, not a replacement for any archived published result. Comparisons to old saved predictions should be made separately only after verifying the same case IDs and protocol. Sample sizes and missing/truncated counts are explicit.

In [ ]:
def build_reports():
 summaries=[];confusions=[];predictions=[]
 for judge in JUDGES:
  for target in TARGETS:
   control_path=RAW/judge/f'{target}_E1_CODE_code_rot0.csv'
   control=pd.read_csv(control_path,dtype={'Case_ID':str}) if control_path.exists() else None
   if control is not None:
    control=control[control.status.eq('ok')][['Case_ID','pred']].rename(columns={'pred':'baseline_pred'})
   for var in variants(target):
    p=RAW/judge/f"{target}_{var['experiment']}_{var['variant']}.csv"
    if not p.exists():continue
    df=pd.read_csv(p,dtype={'Case_ID':str})
    good=df[df.status.eq('ok')].copy()
    labels=[1,2,3] if target=='T4' else [0,1]
    if len(good):
     yt=good.y_true.astype(int);yp=good.pred.astype(int)
     a=accuracy_score(yt,yp);f1=f1_score(yt,yp,labels=labels,average='macro',zero_division=0)
     bal=balanced_accuracy_score(yt,yp)
     qwk=cohen_kappa_score(yt,yp,labels=labels,weights='quadratic') if target=='T4' else float('nan')
     cm=confusion_matrix(yt,yp,labels=labels)
     for i,label in enumerate(labels):
      tp=int(cm[i,i]);support=int(cm[i].sum());predn=int(cm[:,i].sum())
      confusions.append({'judge':judge,'target':target,'experiment':var['experiment'],'variant':var['variant'],
       'class_label':label,'support':support,'predicted_n':predn,'true_positive':tp,
       'recall':tp/support if support else None,'precision':tp/predn if predn else 0})
     stability=float('nan')
     if control is not None:
      joined=good[['Case_ID','pred']].merge(control,on='Case_ID',validate='one_to_one')
      stability=float((joined.pred.astype(int)==joined.baseline_pred.astype(int)).mean()) if len(joined) else float('nan')
    else:a=f1=bal=qwk=stability=float('nan')
    summaries.append({'judge':judge,'target':target,'experiment':var['experiment'],'variant':var['variant'],
      'expected_n':EXPECTED_N[target],'saved_n':len(df),'valid_n':len(good),'excluded_n':len(df)-len(good),
      'complete':len(df)==EXPECTED_N[target] and df.Case_ID.is_unique and len(good)==EXPECTED_N[target],
      'macro_f1':f1,'accuracy':a,'balanced_accuracy':bal,'qwk':qwk,'agreement_with_new_control':stability})
    predictions.append(good.assign(judge=judge,target=target,experiment=var['experiment'],variant=var['variant']))
 summary=pd.DataFrame(summaries);per_class=pd.DataFrame(confusions)
 summary.to_csv(REPORTS/'summary.csv',index=False)
 per_class.to_csv(REPORTS/'per_class.csv',index=False)
 if predictions:pd.concat(predictions,ignore_index=True).to_csv(REPORTS/'all_valid_predictions.csv',index=False)
 meta={'analysis':'Answer-code assignment E1 and option order E2 sensitivity; not architecture comparison',
   'input_hashes':EXPECTED_SHA,'n_expected':EXPECTED_N,'model_profiles':JUDGES,
   'all_complete':len(summary)==sum(len(variants(t)) for t in TARGETS)*len(JUDGES) and summary.complete.all(),
   'date_created_utc':pd.Timestamp.utcnow().isoformat(),'run_id':RUN_ID}
 atomic_write_json(meta,MANIFEST/'report_manifest.json')
 display(summary)
 return summary
summary=build_reports()


In [ ]:
# Export one ZIP of all persisted results. Download is optional; files remain on Drive.
summary=build_reports()
zip_path=ROOT/'results'/(RUN_ID+'_BUNDLE.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
 for p in RUN.rglob('*'):
  if p.is_file():z.write(p,arcname=str(p.relative_to(RUN)))
print('Saved',zip_path,'bytes',zip_path.stat().st_size)
from google.colab import files
files.download(str(zip_path))


## Reporting rules / safeguards

* The two perturbations are **independent**: E1 rotates code-label assignments without moving lines; E2 rotates line positions without changing code-label assignments. These are rotations (not exhaustive permutations), chosen to bound GPU cost. In a two-label target there is one non-identity variant per experiment; in T4 there are two.
* The existing manuscript's FP16 judge results and the new control should be checked for matching revisions, prompt text, token context, and case inclusion before making direct numeric comparisons. New variants are sensitivity diagnostics **within each judge**, not relative performance rankings.
* Full completion requires both judges, every target, and every variant at full expected N with no oversized prompts. `complete=False` flags partial results. Excluded long cases are written with status, never silently omitted.
* Interpretation should emphasize **category recall, predicted-class frequency, macro-F1, ordinal QWK, and agreement with the new control**. A high agreement-to-control cannot alone establish validity. No finding is claimed until the cells have been run and outputs inspected.
* Input is not edited. Never overwrite GOLD or OOF sheets. No learning, training, or architecture comparison occurs.
